In [2]:
import os
import re
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Set, Dict, Any

# Constants
PROJECT_ROOT = "/home/aless/PROJECTS/LLM-Needs-a-Plan"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "src/results")
DATA_DIR = os.path.join(PROJECT_ROOT, "src/data")
VAL_PATH = os.path.join(PROJECT_ROOT, "VAL/build/linux64/Release/bin/Validate")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Results Dir: {RESULTS_DIR}")
print(f"Data Dir: {DATA_DIR}")
print(f"VAL Path: {VAL_PATH}")

ModuleNotFoundError: No module named 'matplotlib'

In [1]:
def get_domain_actions(domain_path: str) -> Set[str]:
    """
    Parse PDDL domain file to extract action names.
    """
    actions = set()
    try:
        with open(domain_path, 'r') as f:
            content = f.read().lower()
            # Simple regex to find (:action action-name
            matches = re.findall(r'\(:action\s+([^\s\)]+)', content)
            actions.update(matches)
    except Exception as e:
        print(f"Error reading domain {domain_path}: {e}")
    return actions

def clean_plan(plan_path: str, valid_actions: Set[str]) -> List[str]:
    """
    Read plan file and filter only lines that start with a valid action.
    """
    cleaned_actions = []
    try:
        with open(plan_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip().lower()
                # Remove comments
                if ';' in line:
                    line = line.split(';')[0].strip()
                
                if not line:
                    continue
                    
                # Check if line looks like (action ...)
                if line.startswith('(') and line.endswith(')'):
                    content = line[1:-1].strip()
                    parts = content.split()
                    if parts and parts[0] in valid_actions:
                        cleaned_actions.append(line)
    except Exception as e:
        print(f"Error reading plan {plan_path}: {e}")
    return cleaned_actions

def validate_plan(domain_path: str, problem_path: str, plan_actions: List[str]) -> Dict[str, Any]:
    """
    Run VAL to validate the plan.
    """
    if not plan_actions:
        return {"valid": False, "error": "Empty plan"}

    # Create a temporary plan file
    temp_plan_path = "temp_plan.txt"
    with open(temp_plan_path, 'w') as f:
        for action in plan_actions:
            f.write(f"{action}\n")

    try:
        # Run VAL
        # Validate domain problem plan
        cmd = [VAL_PATH, "-v", domain_path, problem_path, temp_plan_path]
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        output = result.stdout
        
        is_valid = "Plan valid" in output
        
        # Extract plan length (cost) if valid
        length = len(plan_actions) # Default to number of actions
        
        # VAL output usually contains "Plan valid" or "Plan failed"
        # It might also output the cost.
        
        return {
            "valid": is_valid,
            "length": length,
            "output": output
        }
        
    except Exception as e:
        return {"valid": False, "error": str(e)}
    finally:
        if os.path.exists(temp_plan_path):
            os.remove(temp_plan_path)


NameError: name 'Set' is not defined

In [ ]:
# Main processing loop
results_data = []

# Iterate over models
for model in os.listdir(RESULTS_DIR):
    model_path = os.path.join(RESULTS_DIR, model)
    if not os.path.isdir(model_path):
        continue
        
    print(f"Processing model: {model}")
    
    # Iterate over domains
    for domain in os.listdir(model_path):
        domain_results_path = os.path.join(model_path, domain)
        if not os.path.isdir(domain_results_path):
            continue
            
        print(f"  Processing domain: {domain}")
        
        # Find domain file
        domain_data_path = os.path.join(DATA_DIR, domain)
        domain_file = None
        # Try common names
        candidates = [
            os.path.join(domain_data_path, f"{domain}_domain.pddl"),
            os.path.join(domain_data_path, "domain.pddl"),
            os.path.join(domain_data_path, f"{domain.replace('citycar', 'city_car')}_domain.pddl") # Handle citycar mismatch
        ]
        
        # Also search for any file ending in _domain.pddl
        if os.path.exists(domain_data_path):
             for f in os.listdir(domain_data_path):
                 if f.endswith("_domain.pddl"):
                     candidates.append(os.path.join(domain_data_path, f))

        for cand in candidates:
            if os.path.exists(cand):
                domain_file = cand
                break
        
        if not domain_file:
            print(f"    Could not find domain file for {domain}")
            continue
            
        print(f"    Using domain file: {domain_file}")
        valid_actions = get_domain_actions(domain_file)
        print(f"    Found {len(valid_actions)} valid actions: {valid_actions}")
        
        # Iterate over plans
        for plan_file in os.listdir(domain_results_path):
            if not plan_file.endswith("_plan.txt"):
                continue
                
            # Extract instance name (e.g., instance-01)
            instance_name = plan_file.replace("_plan.txt", "")
            problem_file = os.path.join(domain_data_path, f"{instance_name}.pddl")
            
            if not os.path.exists(problem_file):
                print(f"    Could not find problem file for {instance_name}")
                continue
                
            plan_path = os.path.join(domain_results_path, plan_file)
            
            # Clean and validate
            cleaned_actions = clean_plan(plan_path, valid_actions)
            validation_result = validate_plan(domain_file, problem_file, cleaned_actions)
            
            results_data.append({
                "Model": model,
                "Domain": domain,
                "Problem": instance_name,
                "Valid": validation_result["valid"],
                "Length": validation_result["length"],
                "Raw_Actions": len(cleaned_actions)
            })

print("Processing complete.")

In [ ]:
# Create DataFrame
df = pd.DataFrame(results_data)
print(f"Total records: {len(df)}")
df.head()

In [ ]:
# Analysis: Success Rate
if not df.empty:
    success_rates = df.groupby(['Model', 'Domain'])['Valid'].mean().reset_index()
    success_rates['Success Rate (%)'] = success_rates['Valid'] * 100
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=success_rates, x='Domain', y='Success Rate (%)', hue='Model')
    plt.title('Success Rate by Model and Domain')
    plt.ylim(0, 100)
    plt.grid(axis='y')
    plt.show()
    
    print(success_rates)
else:
    print("No data to plot.")

In [ ]:
# Analysis: Plan Length (only for valid plans)
if not df.empty:
    valid_plans = df[df['Valid'] == True]
    
    if not valid_plans.empty:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=valid_plans, x='Domain', y='Length', hue='Model')
        plt.title('Plan Length Distribution (Valid Plans Only)')
        plt.grid(axis='y')
        plt.show()
    else:
        print("No valid plans to analyze length.")
else:
    print("No data to plot.")